# 01 · From an array to a scientific cube

## Context

A simulation has returned monthly temperature values as a three-dimensional
NumPy array. The numbers are useful, but the axes do not yet say which direction
is time, latitude, or longitude.

## Question

How can we turn those values into a self-describing scientific object and
inspect both its spatial and temporal structure?

## Analysis story

We will name the axes and attach coordinates, units, and provenance. Then we
will compare two familiar slices before sending the same cube through one
minimal plotting pipe into CubeDynamics' interactive viewer.

## Give the array space, time, a name, units, and provenance

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr

from cubedynamics import pipe, verbs as v

# Coordinates turn anonymous array axes into scientific dimensions. Here each
# monthly time step contains a 5-row × 6-column spatial grid.
time = pd.date_range("2025-01-01", periods=18, freq="MS")
y = np.linspace(40.4, 39.6, 5)
x = np.linspace(-105.5, -104.5, 6)

# The singleton axes make NumPy broadcast a temporal signal across space and a
# spatial signal across time. Their sum has shape (time, y, x).
month = np.arange(time.size)[:, None, None]
season = 8 * np.sin(2 * np.pi * month / 12)
spatial = 3 * (y[None, :, None] - y.mean()) - 2 * (x[None, None, :] - x.mean())

# A DataArray pairs values with dimension names, coordinates, a variable name,
# units, and provenance—the minimum useful cube contract for these examples.
cube = xr.DataArray(
    16 + season + spatial,
    dims=("time", "y", "x"),
    coords={"time": time, "y": y, "x": x},
    name="air_temperature",
    attrs={"units": "degC", "source": "deterministic vignette example"},
)

# Assertions double as executable documentation: if the cube contract changes,
# the notebook stops here with a useful failure instead of producing a bad plot.
assert cube.dims == ("time", "y", "x")
cube

## Figure 1 · Read the cube in familiar views

Before using an interactive cube, compare a single map with the history of one
pixel. Both views come from the same labeled object.

In [ ]:
import matplotlib.pyplot as plt

# Two familiar 2D views help participants understand the 3D object: isel chooses
# a slice by integer position; sel chooses one location by coordinate value.
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), constrained_layout=True)
cube.isel(time=6).plot(ax=axes[0], cmap="magma", cbar_kwargs={"label": "°C"})
axes[0].set_title("One spatial slice")
cube.sel(y=y[2], x=x[3]).plot(ax=axes[1], marker="o", color="#2f6f6d")
axes[1].set_title("One pixel through time")
axes[1].set_ylabel("Air temperature (°C)")
plt.show()

## Pipe · Open the repository-native cube viewer

The analytical sentence is deliberately small. `pipe` introduces the value,
`v.plot` says what to do, and `unwrap()` returns the viewer object. Drag the
resulting cube to rotate it and use the wheel or trackpad to zoom.

In [ ]:
from html import escape

from IPython.display import HTML

# The scientific intent stays visible even though the renderer is sophisticated.
viewer = (
    pipe(cube)
    | v.plot(title="Synthetic air-temperature cube", cmap="magma")
).unwrap()

# Escape the viewer's complete HTML document into an iframe srcdoc so it cannot
# interfere with the surrounding documentation header or styles.
assert viewer.data.dims == ("time", "y", "x")
viewer_srcdoc = escape(viewer.to_html(), quote=True)
HTML(
    f'''<iframe
        title="Interactive synthetic air-temperature cube"
        srcdoc="{viewer_srcdoc}"
        style="width: 100%; height: 760px; border: 1px solid #b8c5c2; border-radius: 4px;"
        sandbox="allow-scripts"
        loading="lazy"
    ></iframe>'''
)

## What the figure tells us

The map shows a smooth spatial gradient while the pixel history preserves the
seasonal cycle. The interactive viewer confirms that these are not separate
products: they are different readings of one `(time, y, x)` cube.

## Try the next variation

Change the seasonal amplitude or reverse the `y` coordinates. Which visual
features change, and which parts of the pipe remain identical?